# Basketball YOLO Model Training

농구 중계 영상 분석을 위한 YOLOv8 객체 탐지 모델 학습

**Classes (10개):**
- ball, ball-in-basket, number, player, player-in-possession
- player-jump-shot, player-layup-dunk, player-shot-block, referee, rim

## 1. 환경 설정

In [7]:
!pip install ultralytics -q

In [8]:
import os
import yaml
from pathlib import Path
from ultralytics import YOLO
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
if torch.backends.mps.is_available():
    print("MPS (Apple Silicon) is available")

PyTorch: 2.7.0
CUDA available: False
MPS (Apple Silicon) is available


## 2. 데이터셋 확인

In [9]:
DATASET_DIR = Path("dataset")
DATA_YAML = DATASET_DIR / "data.yaml"

with open(DATA_YAML) as f:
    data_cfg = yaml.safe_load(f)

print("=== 데이터셋 설정 ===")
print(f"클래스 수: {data_cfg['nc']}")
print(f"클래스: {data_cfg['names']}")

for split in ["train", "valid", "test"]:
    img_dir = DATASET_DIR / split / "images"
    count = len(list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.png")))
    print(f"{split}: {count}장")

=== 데이터셋 설정 ===
클래스 수: 10
클래스: ['ball', 'ball-in-basket', 'number', 'player', 'player-in-possession', 'player-jump-shot', 'player-layup-dunk', 'player-shot-block', 'referee', 'rim']
train: 464장
valid: 96장
test: 94장


In [10]:
# data.yaml 경로를 절대경로로 수정 (ultralytics가 경로를 찾을 수 있도록)
abs_dataset = Path(os.getcwd()) / "dataset"

fixed_yaml = {
    "train": str(abs_dataset / "train" / "images"),
    "val":   str(abs_dataset / "valid" / "images"),
    "test":  str(abs_dataset / "test"  / "images"),
    "nc":    data_cfg["nc"],
    "names": data_cfg["names"],
}

FIXED_YAML = Path("data_abs.yaml")
with open(FIXED_YAML, "w") as f:
    yaml.dump(fixed_yaml, f, allow_unicode=True, default_flow_style=False)

print(f"절대경로 yaml 생성: {FIXED_YAML.resolve()}")

절대경로 yaml 생성: /Users/jaehyun/Documents/Projects/DL/BasketballBroadcastAnalysis/data_abs.yaml


## 3. 학습 설정

In [11]:
# 학습 하이퍼파라미터
TRAIN_CONFIG = {
    "model":    "yolov8m.pt",   # n/s/m/l/x 중 선택 (n=가장 빠름, x=가장 정확)
    "epochs":   50,
    "imgsz":    640,
    "batch":    16,             # GPU 메모리에 따라 조정 (부족하면 8로)
    "workers":  4,
    "device":   0 if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu",
    "project":  "runs/train",
    "name":     "basketball",
    "exist_ok": True,
    "mosaic":   1.0,
    "mixup":    0.1,
    "flipud":   0.0,            # 농구는 상하 반전 불필요
    "fliplr":   0.5,
    "lr0":      0.01,
    "lrf":      0.01,
    "optimizer": "AdamW",
    "patience": 15,             # Early stopping
    "save":     True,
    "plots":    True,
}

print("=== 학습 설정 ===")
for k, v in TRAIN_CONFIG.items():
    print(f"  {k}: {v}")

=== 학습 설정 ===
  model: yolov8m.pt
  epochs: 50
  imgsz: 640
  batch: 16
  workers: 4
  device: mps
  project: runs/train
  name: basketball
  exist_ok: True
  mosaic: 1.0
  mixup: 0.1
  flipud: 0.0
  fliplr: 0.5
  lr0: 0.01
  lrf: 0.01
  optimizer: AdamW
  patience: 15
  save: True
  plots: True


## 4. 모델 학습

In [12]:
model_name = TRAIN_CONFIG.pop("model")
model = YOLO(model_name)
print(model.info())

YOLOv8m summary: 169 layers, 25,902,640 parameters, 0 gradients, 79.3 GFLOPs
(169, 25902640, 0, 79.3204224)


In [14]:
results = model.train(
    data=str(FIXED_YAML),
    **TRAIN_CONFIG,
)

New https://pypi.org/project/ultralytics/8.4.46 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.33 🚀 Python-3.10.19 torch-2.7.0 MPS (Apple M4)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data_abs.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=basketball, nbs=64, nms=False, opset=None, optimize

RuntimeError: MPS backend out of memory (MPS allocated: 19.83 GB, other allocations: 304.12 MB, max allowed: 20.13 GB). Tried to allocate 25.00 MB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

## 5. 학습 결과 확인

In [ ]:
from IPython.display import Image, display

save_dir = Path(results.save_dir)
print(f"결과 저장 경로: {save_dir}")

for plot_file in ["results.png", "confusion_matrix.png", "PR_curve.png"]:
    p = save_dir / plot_file
    if p.exists():
        print(f"\n--- {plot_file} ---")
        display(Image(str(p)))

## 6. 검증 (Validation)

In [ ]:
best_weights = save_dir / "weights" / "best.pt"
print(f"Best weights: {best_weights}")

best_model = YOLO(str(best_weights))
val_results = best_model.val(data=str(FIXED_YAML), split="val")

print(f"\n=== Validation 결과 ===")
print(f"mAP50:     {val_results.box.map50:.4f}")
print(f"mAP50-95:  {val_results.box.map:.4f}")
print(f"Precision: {val_results.box.mp:.4f}")
print(f"Recall:    {val_results.box.mr:.4f}")

In [ ]:
print("=== 클래스별 AP50 ===")
for i, name in enumerate(data_cfg["names"]):
    ap = val_results.box.ap50[i] if i < len(val_results.box.ap50) else float("nan")
    print(f"  {name:<30}: {ap:.4f}")

## 7. 테스트 이미지 추론

In [ ]:
import random
import matplotlib.pyplot as plt

test_images = list((abs_dataset / "test" / "images").glob("*.jpg"))
samples = random.sample(test_images, min(4, len(test_images)))

pred_results = best_model.predict(
    source=samples,
    conf=0.25,
    iou=0.45,
    save=True,
    project="runs/predict",
    name="basketball_test",
    exist_ok=True,
)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i, r in enumerate(pred_results):
    img = r.plot()
    axes[i].imshow(img[:, :, ::-1])
    axes[i].set_title(f"Sample {i+1}: {len(r.boxes)}개 탐지")
    axes[i].axis("off")

plt.tight_layout()
plt.savefig("test_predictions.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. 모델 내보내기 (Export)

In [ ]:
# ONNX 형식으로 내보내기 (추론 배포용)
export_path = best_model.export(format="onnx", imgsz=640, simplify=True)
print(f"ONNX 모델 저장: {export_path}")